# 2026 CITS4012 Project
*Make sure you change the file name with your group id.*

# Readme
**Group:** XX  ·  **Task:** MultiRC (multi-sentence reading comprehension, each candidate answer labelled true/false)

**Environment.** Google Colab with a T4 GPU. Runs top to bottom with *Runtime → Run all*. Uses PyTorch, pandas, numpy and matplotlib (preinstalled on Colab); the pretrained baselines also use `transformers`, installed in their own cells.

**Data.** Put `train_456-fixedIds.json` and `test_83-fixedIds.json` (from the unit's shared folder) in a folder called `MultiRC/` next to this notebook, or in `MyDrive/MultiRC/` on Google Drive. Section 1.0 looks in both places and otherwise asks you to upload the two files.

**Splits.** The provided test file is the original MultiRC dev split (the official test set is hidden). We hold out 15% of the training *passages* as a validation set (seed 42). All tuning and model selection use validation only; the test set is used once, for the final numbers. The validation passage IDs are printed in 1.2 and saved to `results/splits.json`.

**Where the numbers come from.** Every result in the report is logged to `results/results.csv` by the cells in this notebook, and every model's predictions are saved to `results/preds/` in one shared format.

**Saved models.** *[TODO: links + loading code for any checkpoints that are too slow to retrain]*

**Notes for the marker.** *[TODO]*

# 1.Dataset Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 1.0 Setup
Imports, seeds, paths, and the shared utilities every section uses: the evaluation metrics (defined here because training uses them for model selection) and the results log.

In [3]:
import json, os, random, re, time, math
from collections import Counter
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42                 # global seed for single runs
SEEDS = [42, 43, 44]      # seeds for repeated runs (main model, ablations)

def set_seed(seed: int = SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

RESULTS_DIR = Path("results"); PREDS_DIR = RESULTS_DIR / "preds"; CKPT_DIR = Path("checkpoints")
for d in (RESULTS_DIR, PREDS_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE, "| torch", torch.__version__)

device: cpu | torch 2.14.1


In [4]:
TRAIN_FILE, TEST_FILE = "train_456-fixedIds.json", "test_83-fixedIds.json"

def find_data_dir() -> Path:
    """Look for the MultiRC files locally, then on Google Drive, then ask for an upload."""
    candidates = [Path("MultiRC"), Path("./MultiRC")]
    for c in candidates:
        if (c / TRAIN_FILE).exists() and (c / TEST_FILE).exists():
            return c
    try:  # on Colab: try Drive, then upload
        from google.colab import drive, files
        drive.mount("/content/drive")
        if (candidates[1] / TRAIN_FILE).exists():
            return candidates[1]
        candidates[0].mkdir(exist_ok=True)
        for name, data in files.upload().items():
            (candidates[0] / name).write_bytes(data)
        return candidates[0]
    except ImportError:
        raise FileNotFoundError(f"Put {TRAIN_FILE} and {TEST_FILE} in ./MultiRC/")

DATA_DIR = find_data_dir()
print("data dir:", DATA_DIR)

data dir: MultiRC


In [5]:
# ---- Shared prediction format ------------------------------------------------
# Every model writes one row per answer option with these columns.
# evaluate() only needs the first four; answer_id and score are for analysis.
PRED_COLUMNS = ["para_id", "question_id", "label", "prediction", "answer_id", "score"]

def evaluate(preds: pd.DataFrame) -> dict:
    """MultiRC metrics, matching the official multirc_measures.py (Khashabi et al., 2018).

    F1a : precision/recall/F1 over all answer options pooled together.
    F1m : precision and recall computed per question (1.0 when the denominator is 0),
          averaged over questions, then F1 = harmonic mean of the two averages.
    EM0 : share of questions where every option is predicted correctly.
    Acc : plain option-level accuracy, for reference only.
    preds must hold ONE split; para_id is the passage index inside its source file.
    """
    keys = ["para_id", "question_id"]
    if "answer_id" in preds:
        assert not preds.duplicated(keys + ["answer_id"]).any(), "duplicate rows: mixed splits?"
    y = preds["label"].astype(bool).to_numpy()
    p = preds["prediction"].astype(bool).to_numpy()

    agree, n_pred, n_true = int((y & p).sum()), int(p.sum()), int(y.sum())
    P = agree / n_pred if n_pred else 1.0
    R = agree / n_true if n_true else 1.0
    f1a = 2 * P * R / (P + R) if P + R else 0.0

    per_q = pd.DataFrame({"agree": y & p, "y": y, "p": p, "wrong": y != p,
                          "para_id": preds["para_id"].to_numpy(),
                          "question_id": preds["question_id"].to_numpy()}).groupby(keys).sum()
    qP = np.where(per_q.p > 0, per_q.agree / per_q.p.clip(lower=1), 1.0)
    qR = np.where(per_q.y > 0, per_q.agree / per_q.y.clip(lower=1), 1.0)
    mP, mR = qP.mean(), qR.mean()
    f1m = 2 * mP * mR / (mP + mR) if mP + mR else 0.0
    em0 = float((per_q.wrong == 0).mean())
    return {"F1a": float(f1a), "F1m": float(f1m), "EM0": em0, "Acc": float((y == p).mean()),
            "P_a": P, "R_a": R, "n_questions": len(per_q), "n_options": len(preds)}

# ---- Results log: the single source for every number in the report ------------
RESULTS_CSV = RESULTS_DIR / "results.csv"

def save_predictions(preds: pd.DataFrame, model: str, split: str, seed=None) -> Path:
    path = PREDS_DIR / f"{model}_{split}" if seed is None else PREDS_DIR / f"{model}_{split}_s{seed}"
    path = path.with_suffix(".csv")
    preds[[c for c in PRED_COLUMNS if c in preds]].to_csv(path, index=False)
    return path

def log_result(model: str, split: str, metrics: dict, seed=None, **extra) -> dict:
    """Append one run to results/results.csv (re-logging the same model/split/seed replaces it)."""
    seed = "-" if seed is None else seed
    row = {"model": model, "split": split, "seed": seed,
           **{k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()},
           **extra, "logged_at": time.strftime("%Y-%m-%d %H:%M")}
    old = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
    if len(old):
        same = (old["model"] == model) & (old["split"] == split) & \
               (old["seed"].astype(str) == str(seed))
        old = old[~same]
    pd.concat([old, pd.DataFrame([row])], ignore_index=True).to_csv(RESULTS_CSV, index=False)
    return row

## 1.1 Load and flatten
One row per answer option. `para_id` is the passage's index inside its source file (so it lines up with prediction files), `question_id` is the question's index in its passage, `answer_id` the option's index in its question. Sentences come from MultiRC's own `Sent N:` markup, so no sentence splitter is needed.

In [6]:
SENT_RE = re.compile(r"<b>Sent \d+: </b>(.*?)<br>", re.S)

def split_sentences(html: str) -> list:
    return [re.sub(r"\s+", " ", s).strip() for s in SENT_RE.findall(html)]

def load_multirc(path: Path, source: str) -> pd.DataFrame:
    rows = []
    for para_id, item in enumerate(json.load(open(path))["data"]):
        sents = split_sentences(item["paragraph"]["text"])
        for q in item["paragraph"]["questions"]:
            for answer_id, a in enumerate(q["answers"]):
                rows.append({
                    "source": source, "para_id": para_id, "doc_id": item["id"],
                    "question_id": int(q["idx"]), "answer_id": answer_id,
                    "sentences": sents, "passage": " ".join(sents),
                    "question": q["question"], "answer": a["text"], "label": bool(a["isAnswer"]),
                    "multisent": q.get("multisent"), "sentences_used": q.get("sentences_used"),
                })
    return pd.DataFrame(rows)

train_full = load_multirc(DATA_DIR / TRAIN_FILE, "train")
test_df = load_multirc(DATA_DIR / TEST_FILE, "test")
print(f"train file: {len(train_full):,} options | test file: {len(test_df):,} options")
train_full.head(3)[["para_id", "question_id", "answer_id", "question", "answer", "label"]]

train file: 27,243 options | test file: 4,848 options


,para_id,question_id,answer_id,question,answer,label
0,0,0,0,Does the author claim the animated films messa...,Yes,True
1,0,0,1,Does the author claim the animated films messa...,"Uphold, and continue",True
2,0,0,2,Does the author claim the animated films messa...,No,False


## 1.2 Train / validation split
15% of training **passages** go to validation, so no passage appears in both. Fixed seed, saved to `results/splits.json`. Everyone uses this split.

In [7]:
VAL_FRACTION, SPLIT_SEED = 0.15, 42

train_paras = sorted(train_full["para_id"].unique())
val_paras = sorted(random.Random(SPLIT_SEED).sample(train_paras, round(VAL_FRACTION * len(train_paras))))

train_full["split"] = np.where(train_full["para_id"].isin(val_paras), "val", "train")
test_df["split"] = "test"
train_df = train_full[train_full["split"] == "train"].reset_index(drop=True)
val_df = train_full[train_full["split"] == "val"].reset_index(drop=True)

assert not set(train_df.para_id) & set(val_df.para_id)
json.dump({"seed": SPLIT_SEED, "val_fraction": VAL_FRACTION, "val_para_ids": [int(i) for i in val_paras]},
          open(RESULTS_DIR / "splits.json", "w"))
print(f"{len(val_paras)} validation passages:", [int(i) for i in val_paras])

68 validation passages: [3, 12, 13, 15, 16, 22, 40, 44, 47, 49, 52, 57, 63, 71, 79, 81, 101, 110, 111, 112, 114, 119, 125, 135, 140, 142, 150, 172, 174, 176, 183, 185, 193, 194, 214, 216, 229, 235, 258, 274, 279, 282, 287, 301, 302, 308, 309, 316, 321, 327, 332, 346, 357, 359, 366, 373, 377, 379, 388, 390, 412, 413, 414, 424, 433, 441, 445, 453]


## 1.3 Dataset statistics
Feeds the split statistics table in report Section 3.1.

In [8]:
def split_stats(df):
    paras = df.drop_duplicates(["para_id"])
    return {"passages": df.para_id.nunique(),
            "questions": df.groupby(["para_id", "question_id"]).ngroups,
            "options": len(df),
            "% options true": round(100 * df.label.mean(), 1),
            "options / question": round(len(df) / df.groupby(["para_id", "question_id"]).ngroups, 2),
            "sentences / passage": round(paras.sentences.map(len).mean(), 1)}

pd.DataFrame({name: split_stats(d) for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]})

,train,val,test
passages,388.00,68.00,83.00
questions,4354.00,777.00,953.00
options,22959.00,4284.00,4848.00
% options true,44.10,44.40,42.80
options / question,5.27,5.51,5.09
sentences / passage,14.50,13.70,14.20


## 1.4 Tokenisation and vocabulary
Lower-cased word/punctuation tokens. The vocabulary is built from the **training split only**; words seen fewer than `MIN_FREQ` times map to `<unk>`.

In [9]:
TOKEN_RE = re.compile(r"\w+|[^\w\s]")
PAD, UNK = "<pad>", "<unk>"
MIN_FREQ = 2

def tokenize(text: str) -> list:
    return TOKEN_RE.findall(text.lower())

class Vocab:
    def __init__(self, counter: Counter, min_freq: int = MIN_FREQ):
        self.itos = [PAD, UNK] + sorted(w for w, c in counter.items() if c >= min_freq)
        self.stoi = {w: i for i, w in enumerate(self.itos)}
    def __len__(self): return len(self.itos)
    def encode(self, tokens): return [self.stoi.get(t, 1) for t in tokens]

def build_vocab(df: pd.DataFrame) -> Vocab:
    counter = Counter()
    for passage in df.drop_duplicates("para_id")["passage"]:
        counter.update(tokenize(passage))
    for text in pd.concat([df.drop_duplicates(["para_id", "question_id"])["question"], df["answer"]]):
        counter.update(tokenize(text))
    return Vocab(counter)

vocab = build_vocab(train_df)
def oov_rate(df):
    toks = [t for p in df.drop_duplicates("para_id").passage for t in tokenize(p)]
    return sum(vocab.stoi.get(t, 1) == 1 for t in toks) / len(toks)
print(f"vocab size {len(vocab):,} | passage OOV rate: val {oov_rate(val_df):.1%}, test {oov_rate(test_df):.1%}")

vocab size 10,972 | passage OOV rate: val 9.6%, test 9.1%


## 1.5 Encoding and batches
Each example is one (passage, question, candidate answer) triple. Passages are truncated to `MAX_PASSAGE_LEN` tokens (covers about 95% of training passages); questions and answers to `MAX_QA_LEN`.

Every model receives batches in this format:

| key | shape | dtype | meaning |
|---|---|---|---|
| `passage` | [B, Lp] | long | passage token ids, 0 = padding |
| `passage_mask` | [B, Lp] | bool | True for real tokens |
| `passage_sent` | [B, Lp] | long | sentence index of each token, −1 for padding |
| `n_sents` | [B] | long | number of (kept) sentences |
| `question`, `question_mask` | [B, Lq] | long, bool | question tokens |
| `answer`, `answer_mask` | [B, La] | long, bool | candidate answer tokens |
| `label` | [B] | float | 1.0 if the option is correct |
| `meta` | list of B tuples | – | (para_id, question_id, answer_id), used to write predictions |

In [10]:
MAX_PASSAGE_LEN, MAX_QA_LEN = 500, 40

def encode_text(text, max_len):
    ids = vocab.encode(tokenize(text))[:max_len]
    return ids or [1]                      # never empty (a few answers have no word tokens)

def encode_passage(sentences, max_len=MAX_PASSAGE_LEN):
    ids, sent_idx = [], []
    for s, sent in enumerate(sentences):
        toks = vocab.encode(tokenize(sent))[: max_len - len(ids)]
        ids += toks; sent_idx += [s] * len(toks)
        if len(ids) >= max_len:
            break
    return ids, sent_idx

class MultiRCDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        cache = {}
        self.items = []
        for r in df.itertuples(index=False):
            key = (r.source, r.para_id)
            if key not in cache:
                cache[key] = encode_passage(r.sentences)
            p_ids, p_sent = cache[key]
            self.items.append((p_ids, p_sent, encode_text(r.question, MAX_QA_LEN),
                               encode_text(r.answer, MAX_QA_LEN), float(r.label),
                               (int(r.para_id), int(r.question_id), int(r.answer_id))))
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def _pad(seqs, value=0):
    out = torch.full((len(seqs), max(map(len, seqs))), value, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, : len(s)] = torch.tensor(s, dtype=torch.long)
    return out

def collate(batch):
    p, ps, q, a, y, meta = zip(*batch)
    passage, question, answer = _pad(p), _pad(q), _pad(a)
    return {"passage": passage, "passage_mask": passage != 0, "passage_sent": _pad(ps, -1),
            "n_sents": torch.tensor([max(s) + 1 for s in ps]),
            "question": question, "question_mask": question != 0,
            "answer": answer, "answer_mask": answer != 0,
            "label": torch.tensor(y), "meta": list(meta)}

def make_loader(df, batch_size=32, shuffle=False, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(MultiRCDataset(df), batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate, generator=g)

def to_device(batch):
    return {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}

_b = next(iter(make_loader(val_df, batch_size=4)))
print({k: (tuple(v.shape) if torch.is_tensor(v) else v[:2]) for k, v in _b.items()})

{'passage': (4, 291), 'passage_mask': (4, 291), 'passage_sent': (4, 291), 'n_sents': (4,), 'question': (4, 11), 'question_mask': (4, 11), 'answer': (4, 8), 'answer_mask': (4, 8), 'label': (4,), 'meta': [(3, 0, 0), (3, 0, 1)]}


# 2. Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 2.0 Model interface
Every trained model (main model, ablation variants, any from-scratch baseline) follows this contract so the same training, prediction, ablation and attention-plotting code works for all of them:

- built from a `ModelConfig`;
- `forward(batch)` takes the batch dict from 1.5 and returns `{"logits": [B], "attn": {name: weights}}`;
- attention weights keep the shape of what they attend over, e.g. `"passage": [B, Lp]`, `"sentence": [B, S]`.

`ModelConfig` holds one switch per ablation. The switches below are candidates; the main-model owner keeps the ones that match the final design, and each model states which ones it reads.

In [11]:
@dataclass
class ModelConfig:
    vocab_size: int
    emb_dim: int = 100
    hidden_dim: int = 128
    num_layers: int = 1
    dropout: float = 0.3
    rnn_type: str = "lstm"            # "lstm" | "gru"
    shared_encoder: bool = True       # one encoder for passage, question and answer
    use_attention: bool = True        # False -> mean pooling instead of attention
    attn_score: str = "additive"      # "additive" | "dot" | "bilinear"
    use_sentence_attention: bool = True
    fusion: str = "gate"              # "gate" | "concat"
    pooling: str = "attn"             # "attn" | "mean" | "max"

class BaseQAModel(nn.Module):
    """Parent class for every trained model. Subclasses implement forward()."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
    def forward(self, batch: dict) -> dict:
        raise NotImplementedError
    def n_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

def masked_softmax(scores, mask, dim=-1):
    return torch.softmax(scores.masked_fill(~mask, float("-inf")), dim=dim)

def masked_mean(h, mask):
    m = mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1)

## 2.1 Training and prediction
Shared by every trained model. Binary cross-entropy on the logits; the checkpoint with the best **validation F1a** is kept (early stopping on the same metric). Each epoch prints a log line, which stays in the saved outputs.

In [12]:
@dataclass
class TrainConfig:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    patience: int = 3                 # epochs without val F1a improvement before stopping
    weight_decay: float = 0.0
    grad_clip: float = 5.0
    pos_weight: float = None          # optional class weight for true options
    threshold: float = 0.5
    seed: int = SEED
    max_train_batches: int = None     # only for quick smoke tests

@torch.no_grad()
def predict(model, df, batch_size=64, threshold=0.5) -> pd.DataFrame:
    model.eval()
    rows = []
    for batch in make_loader(df, batch_size):
        probs = torch.sigmoid(model(to_device(batch))["logits"]).cpu().tolist()
        for (pid, qid, aid), pr, y in zip(batch["meta"], probs, batch["label"].tolist()):
            rows.append((pid, qid, bool(y), pr >= threshold, aid, pr))
    return pd.DataFrame(rows, columns=PRED_COLUMNS)

def train_model(model, train_df, val_df, tcfg: TrainConfig, name="model"):
    set_seed(tcfg.seed)
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=tcfg.lr, weight_decay=tcfg.weight_decay)
    pw = None if tcfg.pos_weight is None else torch.tensor(tcfg.pos_weight, device=DEVICE)
    loader = make_loader(train_df, tcfg.batch_size, shuffle=True, seed=tcfg.seed)
    best, best_state, bad, history = -1.0, None, 0, []
    for epoch in range(1, tcfg.epochs + 1):
        model.train(); t0 = time.time(); total, n = 0.0, 0
        for i, batch in enumerate(loader):
            if tcfg.max_train_batches and i >= tcfg.max_train_batches:
                break
            batch = to_device(batch)
            loss = F.binary_cross_entropy_with_logits(model(batch)["logits"], batch["label"], pos_weight=pw)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), tcfg.grad_clip); opt.step()
            total += loss.item() * len(batch["label"]); n += len(batch["label"])
        val = evaluate(predict(model, val_df, threshold=tcfg.threshold))
        history.append({"epoch": epoch, "train_loss": total / n, **{f"val_{k}": val[k] for k in ("F1a", "F1m", "EM0")}})
        print(f"[{name}] epoch {epoch:2d} | loss {total / n:.4f} | val F1a {val['F1a']:.4f} "
              f"F1m {val['F1m']:.4f} EM0 {val['EM0']:.4f} | {time.time() - t0:.0f}s")
        if val["F1a"] > best:
            best, bad = val["F1a"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= tcfg.patience:
                print(f"[{name}] early stop, best val F1a {best:.4f}"); break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

## 2.2 Reference model (scaffold, delete before submission)
A small BiLSTM with question/answer-aware attention over passage tokens. It exists only to test the pipeline end to end and to give the ablation and attention-plotting code something to run on until the main model is frozen. It reads `rnn_type`, `use_attention` and `attn_score`.

In [13]:
class ReferenceAttnModel(BaseQAModel):
    def __init__(self, cfg: ModelConfig):
        super().__init__(cfg)
        H = 2 * cfg.hidden_dim
        self.emb = nn.Embedding(cfg.vocab_size, cfg.emb_dim, padding_idx=0)
        rnn = nn.LSTM if cfg.rnn_type == "lstm" else nn.GRU
        self.enc = rnn(cfg.emb_dim, cfg.hidden_dim, cfg.num_layers, batch_first=True, bidirectional=True)
        self.query = nn.Linear(2 * H, H)
        self.W_h, self.W_q, self.v = nn.Linear(H, H, bias=False), nn.Linear(H, H), nn.Linear(H, 1, bias=False)
        self.W_b = nn.Linear(H, H, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
        self.out = nn.Sequential(nn.Linear(3 * H, H), nn.ReLU(), nn.Dropout(cfg.dropout), nn.Linear(H, 1))

    def encode(self, ids, mask):
        lengths = mask.sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(self.drop(self.emb(ids)), lengths,
                                                   batch_first=True, enforce_sorted=False)
        h, _ = self.enc(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=ids.size(1))
        return h

    def score(self, h, q):                      # h [B,L,H], q [B,H] -> [B,L]
        if self.cfg.attn_score == "dot":
            return (h @ q.unsqueeze(-1)).squeeze(-1) / math.sqrt(h.size(-1))
        if self.cfg.attn_score == "bilinear":
            return (self.W_b(h) @ q.unsqueeze(-1)).squeeze(-1)
        return self.v(torch.tanh(self.W_h(h) + self.W_q(q).unsqueeze(1))).squeeze(-1)

    def forward(self, batch):
        P = self.encode(batch["passage"], batch["passage_mask"])
        q = masked_mean(self.encode(batch["question"], batch["question_mask"]), batch["question_mask"])
        a = masked_mean(self.encode(batch["answer"], batch["answer_mask"]), batch["answer_mask"])
        query = torch.tanh(self.query(torch.cat([q, a], -1)))
        attn = {}
        if self.cfg.use_attention:
            alpha = masked_softmax(self.score(P, query), batch["passage_mask"])
            ctx = (alpha.unsqueeze(-1) * P).sum(1)
            attn["passage"] = alpha
        else:
            ctx = masked_mean(P, batch["passage_mask"])
        logits = self.out(self.drop(torch.cat([ctx, query, ctx * query], -1))).squeeze(-1)
        return {"logits": logits, "attn": attn}

## 2.3 Main model
*[Owner B: main model class (subclass `BaseQAModel`), architecture notes, and the final training runs over `SEEDS`, logged with `log_result`.]*

In [ ]:
# TODO (B): class MainModel(BaseQAModel): ...

## 2.4 Baselines
*[Owner A. Each baseline writes predictions in the shared format with `save_predictions` and logs test metrics with `log_result`. Prompt or hyperparameter choices are made on validation only.]*

### 2.4.1 Fine-tuned pretrained transformer

In [ ]:
# TODO (A): DistilBERT / BERT-base sequence-pair classification

### 2.4.2 Open-source LLM, few-shot (Qwen3-4B-Instruct, 2-shot)

In [ ]:
# TODO (A): move the prompting code from LLM_model.ipynb here; write qwen3-4b-2shot_test.csv via save_predictions
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"  # T4 doesn't support bf16
)
model.eval()

SYSTEM = (
    "You are evaluating answers to reading comprehension questions. "
    "Given a paragraph, a question and a candidate answer, decide whether the "
    "candidate answer is correct using only information in the paragraph. "
    "A question may have more than one correct answer. "
    "Reply with exactly one word: True or False."
)

def user_msg(paragraph, question, answer):
    return (f"Paragraph: {paragraph}\n\n"
            f"Question: {question}\n\n"
            f"Candidate answer: {answer}\n\n"
            "Is the candidate answer correct?")

SHOTS = [
    (
        r"""Animated history of the US. Of course the cartoon is highly oversimplified, and most critics consider it one of the weakest parts of the film. But it makes a valid claim which you ignore entirely: That the strategy to promote "gun rights" for white people and to outlaw gun possession by black people was a way to uphold racism without letting an openly terrorist organization like the KKK flourish. Did the 19th century NRA in the southern states promote gun rights for black people? I highly doubt it. But if they didn't, one of their functions was to continue the racism of the KKK. This is the key message of this part of the animation, which is again being ignored by its critics. Buell shooting in Flint. You write: "Fact: The little boy was the class thug, already suspended from school for stabbing another kid with a pencil, and had fought with Kayla the day before". This characterization of a six-year-old as a pencil-stabbing thug is exactly the kind of hysteria that Moore's film warns against. It is the typical right-wing reaction which looks for simple answers that do not contradict the Republican mindset. The kid was a little bastard, and the parents were involved in drugs -- case closed. But why do people deal with drugs? Because it's so much fun to do so? It is by now well documented that the CIA tolerated crack sales in US cities to fund the operation of South American "contras" It is equally well known that the so-called "war on drugs" begun under the Nixon administration is a failure which has cost hundreds of billions and made America the world leader in prison population (both in relative and absolute numbers)""",
        r"""Does the author claim the animated films message is that the NRA upholds racism?""",
        r"""Yes""",
        True
    ),
    (
        r"""Hotel California My first thought: I was going crazy. Twenty-four hours of silence (vacuum, remember); was I hallucinating noises now? I heard it again. It was a fine bell, reminiscent of ancient stone churches and the towering cathedrals I'd seen in documentaries. And accompanying the bell, I saw a light. Now, there were two things here that made ridiculously small amounts of sense. First, the whole in-a-vacuum why's-there-a-bell thing. Second, I was floating in the dark remnants of my broken ship, and any conceivable light sources were not within view; starlight is a distinctly different color and significantly less bright. These signals were the heralds of my saviors. The first words they said to me meant nothing. I wasn't listening; I didn't care; I was going to live; I was going to keep breathing. Next to those, nothing else mattered. The recycled air tasted sweet in my mouth, and all thoughts that crossed my mind were cheap metaphors about life-giving substances and how breathing was like sex, only better. (I reserved the right to revise this opinion later.) When I was done mentally exclaiming over my impossible rescue, I looked around. The ship, it was odd and old, either so outdated or so heavily modified that I couldn't tell what make it was, and somehow, the crew standing around me fit the same description, a singularly atypical amalgamation of folk. And me, I guess I was one more piece in their puzzle. I was one more scrap to weld onto the rest, one more stranded survivor who was found. I was now one of them.""",
        r"""Who is the "Them" the writer refers to being one of?""",
        r"""He felt he was one more person for who nothing else mattered""",
        False
    )
]

def build_messages(paragraph, question, answer, shots=SHOTS):
    msgs = [{"role": "system", "content": SYSTEM}]
    for p, q, a, label in shots:
        msgs.append({"role": "user", "content": user_msg(p, q, a)})
        msgs.append({"role": "assistant", "content": "True" if label else "False"})
    msgs.append({"role": "user", "content": user_msg(paragraph, question, answer)})
    return msgs

# token ids for the two possible answers
TRUE_ID = tok.encode("True", add_special_tokens=False)
FALSE_ID = tok.encode("False", add_special_tokens=False)
assert len(TRUE_ID) == 1 and len(FALSE_ID) == 1, "check tokenisation"
TRUE_ID, FALSE_ID = TRUE_ID[0], FALSE_ID[0]

@torch.no_grad()
def predict(paragraph, question, answer, shots=SHOTS):
    prompt = tok.apply_chat_template(
        build_messages(paragraph, question, answer, shots),
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1]
    return bool(logits[TRUE_ID] > logits[FALSE_ID])

import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/MultiRC/'

df = pd.read_json(DATA_PATH + "test_83-fixedIds.json")


def get_paragraph_and_questions(df):
    for row in df.itertuples():
        para_id = row[0]
        paragraph = row[1]['paragraph']
        paragraph_text = paragraph['text']
        paragraph_questions = paragraph['questions']
        for question in paragraph_questions:
            question_text = question['question']
            question_id = question['idx']                                        # ADDED
            for answer in question['answers']:
                answer_text = answer['text']
                is_answer = answer['isAnswer']
                yield para_id, question_id, paragraph_text, question_text, answer_text, is_answer   # CHANGED
import re

def clean_paragraph(text):
    text = re.sub(r"<b>Sent \d+: </b>", "", text)  # remove "Sent N:" markers
    text = text.replace("<br>", " ")                # line breaks -> spaces
    text = re.sub(r"\s+", " ", text).strip()        # tidy whitespace
    return text

questions_answers = get_paragraph_and_questions(df)

output_file = DATA_PATH + "qwen3_4b_2shot_test_preds.csv"                        # ADDED

#run model
with open(output_file, "w", encoding="utf-8") as f:
  f.write("para_id,question_id,label,prediction,correct\n")                     # ADDED
  try:
      while True:
          para_id, question_id, paragraph, question, answer, label = next(questions_answers)   # CHANGED
          paragraph = clean_paragraph(paragraph)
          prediction = predict(paragraph, question, answer)
          if prediction == label:
            correct = True
          else:
            correct = False
          f.write(f"{para_id},{question_id},{label},{prediction},{correct}\n")  # CHANGED
          f.flush()                                                              # ADDED

  except StopIteration:
      print("Reached the end of the generator!")

### 2.4.3 Constant predictors
Predicting every option false (the majority class) or every option true shows what each metric looks like with no reading at all.

In [ ]:
for name, value in [("all-false", False), ("all-true", True)]:
    preds = test_df[["para_id", "question_id", "label", "answer_id"]].assign(prediction=value, score=float(value))
    save_predictions(preds, name, "test")
    print(name, {k: round(v, 4) for k, v in log_result(name, "test", evaluate(preds)).items() if k in ("F1a", "F1m", "EM0", "Acc")})

all-false {'F1a': 0.0, 'F1m': 0.0063, 'EM0': 0.0031, 'Acc': 0.572}
all-true {'F1a': 0.5995, 'F1m': 0.61, 'EM0': 0.0084, 'Acc': 0.428}


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 3.0 Metrics
F1a, F1m and EM0 are implemented in `evaluate()` in Section 1.0 (training uses them for model selection, so they are defined first). Definitions follow the official MultiRC evaluation script.

## 3.1 Check: existing Qwen3-4B predictions
Runs the shared metrics on the predictions already produced by the LLM baseline, and checks the file lines up with the test set. *[Remove once 2.4.2 produces these predictions inside this notebook.]*

In [ ]:
qwen_path = DATA_DIR / "qwen3_4b_2shot_test_preds.csv"
if qwen_path.exists():
    qwen = pd.read_csv(qwen_path)
    expected = test_df.groupby(["para_id", "question_id"]).size()
    got = qwen.groupby(["para_id", "question_id"]).size()
    assert expected.equals(got), "prediction file does not line up with the test set"
    assert (qwen["label"].to_numpy() == test_df["label"].to_numpy()).all(), "row order differs from test_df"
    qwen["answer_id"] = test_df["answer_id"].to_numpy()
    print({k: round(v, 4) for k, v in evaluate(qwen).items() if k in ("F1a", "F1m", "EM0", "Acc")})
else:
    print("no Qwen prediction file found in", DATA_DIR)

{'F1a': 0.8228, 'F1m': 0.8453, 'EM0': 0.5509, 'Acc': 0.8614}


## 3.2 Smoke test (delete before submission)
Trains the reference model briefly and evaluates it on validation, to show that data → model → training → predictions → metrics → results log all connect. On a Colab GPU this takes about a minute.

In [ ]:
cfg = ModelConfig(vocab_size=len(vocab))
smoke = ReferenceAttnModel(cfg)
smoke, hist = train_model(smoke, train_df, val_df,
                          TrainConfig(epochs=1, max_train_batches=60), name="smoke")
val_preds = predict(smoke, val_df)
print({k: round(v, 4) for k, v in evaluate(val_preds).items() if k in ("F1a", "F1m", "EM0", "Acc")},
      "| params:", f"{smoke.n_params():,}")
_out = smoke(to_device(next(iter(make_loader(val_df, batch_size=2)))))
print("attention keys/shapes:", {k: tuple(v.shape) for k, v in _out["attn"].items()})

[smoke] epoch  1 | loss 0.6684 | val F1a 0.2775 F1m 0.2788 EM0 0.0335 | 135s


{'F1a': 0.2775, 'F1m': 0.2788, 'EM0': 0.0335, 'Acc': 0.5843} | params: 1,858,289


attention keys/shapes: {'passage': (2, 291)}


## 3.3 Overall comparison
*[Owner A: build the report's main table from `results/results.csv` (test split, mean ± std over seeds).]*

In [ ]:
pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else "no results logged yet"

,model,split,seed,F1a,F1m,EM0,Acc,P_a,R_a,n_questions,n_options,logged_at
0,all-false,test,-,0.0000,0.0063,0.0031,0.572,1.000,0.0,953,4848,2026-10-06 17:06
1,all-true,test,-,0.5995,0.6100,0.0084,0.428,0.428,1.0,953,4848,2026-10-06 17:06


## 3.4 Ablation studies
*[Owner C: one run per variant per seed, each changing only the `ModelConfig` switch under test; logged with `log_result(model="main-<variant>", ...)`.]*

In [ ]:
# TODO (C)

## 3.5 Attention analysis
*[Owner C: heatmaps from `model(batch)["attn"]` for correct and incorrect test predictions, shown with passage excerpt, question, answer, gold label and prediction.]*

In [ ]:
# TODO (C)